# N=64 no-augmentation memorization quick-check

This notebook checks whether the `nf_gen_nick_u128_d2p06_noaug_mem100k` run actually memorizes/copies its tiny training set.

The important diagnostic is not `generated[0]` by eye. For every generated sample, compute its nearest training slice and plot the closest matches.

Quick sample while the full sample job is running:

```bash
cd /home/jiamingp/diffusion_models_repo
sbatch -A huterer2 --export=ALL,NUM_SAMPLES=16,BATCH_SIZE=4,SEED=999 scripts/slurm/sample_nf_generalize_n64_memorize.sbatch
```

The quick sample writes a separate `seed999` file, so it should not collide with the full default `seed123` sample.


In [ ]:
from __future__ import annotations

import json
import math
import os
import sys
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_DIR = Path(os.environ.get("PROJECT_DIR", "/home/jiamingp/diffusion_models_repo")).resolve()
if not PROJECT_DIR.exists():
    PROJECT_DIR = Path.cwd().resolve()
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from simdiff_eval.io import load_real_from_config, as_nchw
from simdiff_eval.metrics import field_histogram, batch_power_spectra, power_spectrum_summary

SWEEP_NAME = "nf_generalize_n64_memorize"
RUN_NAME = "nf_gen_nick_u128_d2p06_noaug_mem100k"
CONFIG_PATH = PROJECT_DIR / "local" / SWEEP_NAME / "configs" / f"{RUN_NAME}.yaml"
MANIFEST_PATH = PROJECT_DIR / "local" / SWEEP_NAME / "manifest.json"
SAMPLE_ROOT = PROJECT_DIR / "results" / SWEEP_NAME / "samples"
OUTPUT_DIR = PROJECT_DIR / "results" / SWEEP_NAME / "quickcheck"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_DIR", PROJECT_DIR)
print("CONFIG_PATH", CONFIG_PATH)
print("SAMPLE_ROOT", SAMPLE_ROOT)


In [ ]:
with MANIFEST_PATH.open() as f:
    manifest = json.load(f)
row = next(r for r in manifest if r["run_name"] == RUN_NAME)

display(pd.DataFrame([row])[ [
    "run_name", "dataset_tag", "dataset_size", "n_train_simulations",
    "epochs", "steps_per_epoch", "actual_updates", "batch_size",
] ])

display(pd.DataFrame(row["source_counts"]))


## Load the 64 training slices

This is small, so we load the complete N=64 training set. There is no augmentation in this run, so exact memorization should show up as a generated sample very close to one of these slices.


In [ ]:
real = as_nchw(load_real_from_config(CONFIG_PATH))
print("real", real.shape, float(real.min()), float(real.max()), float(real.mean()), float(real.std()))
assert len(real) == 64, len(real)


In [ ]:
sample_files = sorted(SAMPLE_ROOT.glob(f"{RUN_NAME}_seed*_raw_train_full.npz"))
print("sample files:")
for p in sample_files:
    print(" ", p.name, p.stat().st_size / 1024**2, "MB")

if not sample_files:
    print("No samples found yet. For a quick check run:")
    print("sbatch -A huterer2 --export=ALL,NUM_SAMPLES=16,BATCH_SIZE=4,SEED=999 scripts/slurm/sample_nf_generalize_n64_memorize.sbatch")
    print("For the full sample run:")
    print("sbatch -A huterer2 scripts/slurm/sample_nf_generalize_n64_memorize.sbatch")


In [ ]:
def load_npz_array(path: Path) -> np.ndarray:
    with np.load(path) as data:
        keys = list(data.keys())
        preferred = ["samples", "images", "arr_0", "x", "generated"]
        for key in preferred:
            if key in data:
                return as_nchw(np.asarray(data[key], dtype=np.float32))
        # choose the first array-like key with at least 3 dims
        for key in keys:
            arr = np.asarray(data[key])
            if arr.ndim >= 3:
                return as_nchw(arr.astype(np.float32))
        raise KeyError(f"No image array found in {path}; keys={keys}")

# Prefer full seed123 if present; otherwise use the newest/available quick file.
preferred = [p for p in sample_files if "seed123" in p.name]
SAMPLE_PATH = preferred[0] if preferred else (sample_files[-1] if sample_files else None)

if SAMPLE_PATH is not None:
    generated = load_npz_array(SAMPLE_PATH)
    print("using", SAMPLE_PATH)
    print("generated", generated.shape, float(generated.min()), float(generated.max()), float(generated.mean()), float(generated.std()))
else:
    generated = np.empty((0, 1, 128, 128), dtype=np.float32)


## Raw image panel

This panel is only a sanity check. It intentionally shows multiple training slices and multiple generated samples, not just index 0.


In [ ]:
def plot_grid(images: np.ndarray, title: str, n: int = 16, out_name: str | None = None):
    if len(images) == 0:
        print("no images for", title)
        return
    n = min(n, len(images))
    ncols = min(8, n)
    nrows = math.ceil(n / ncols)
    vals = images[:n, 0]
    vmin, vmax = np.nanpercentile(vals, [1, 99])
    fig, axes = plt.subplots(nrows, ncols, figsize=(2.0 * ncols, 2.15 * nrows), squeeze=False)
    for ax in axes.ravel():
        ax.axis("off")
    for i, ax in enumerate(axes.ravel()[:n]):
        ax.imshow(images[i, 0], origin="lower", cmap="viridis", vmin=vmin, vmax=vmax)
        ax.set_title(str(i), fontsize=9)
    fig.suptitle(title)
    fig.tight_layout()
    if out_name:
        out = OUTPUT_DIR / out_name
        fig.savefig(out, dpi=180, bbox_inches="tight")
        print("wrote", out)
    plt.show()

plot_grid(real, "training slices", n=16, out_name="n64_memorize_training_slices.png")
plot_grid(generated, f"generated samples: {SAMPLE_PATH.name if SAMPLE_PATH else 'missing'}", n=16, out_name="n64_memorize_generated_samples.png")


## Generated-to-training nearest neighbors

For each generated sample `x_j`, compute distance to every training slice `y_i`, then keep the closest training slice.

If this run memorized exact training images, the best rows should have very small `mse`/`mae`, cosine similarity near 1, and the generated/training panels should look nearly identical.


In [ ]:
def flattened_unit(images: np.ndarray) -> np.ndarray:
    flat = np.asarray(images, dtype=np.float64).reshape(len(images), -1)
    flat = flat - flat.mean(axis=1, keepdims=True)
    norm = np.linalg.norm(flat, axis=1, keepdims=True)
    return flat / np.clip(norm, 1e-30, None)


def nearest_training_table(real: np.ndarray, generated: np.ndarray, chunk: int = 32) -> pd.DataFrame:
    if len(generated) == 0:
        return pd.DataFrame()
    real_flat = np.asarray(real, dtype=np.float64).reshape(len(real), -1)
    gen_flat = np.asarray(generated, dtype=np.float64).reshape(len(generated), -1)
    real_unit = flattened_unit(real)
    gen_unit = flattened_unit(generated)

    rows = []
    for start in range(0, len(generated), chunk):
        stop = min(start + chunk, len(generated))
        g = gen_flat[start:stop]
        # MSE distances in chunks: shape (chunk, n_real)
        diff = g[:, None, :] - real_flat[None, :, :]
        mse = np.mean(diff * diff, axis=2)
        nn_idx = np.argmin(mse, axis=1)
        cos = gen_unit[start:stop] @ real_unit.T
        for local_j, i in enumerate(nn_idx):
            j = start + local_j
            d = generated[j, 0] - real[i, 0]
            rows.append({
                "generated_index": j,
                "nearest_train_index": int(i),
                "mse": float(np.mean(d * d)),
                "rmse": float(np.sqrt(np.mean(d * d))),
                "mae": float(np.mean(np.abs(d))),
                "max_abs": float(np.max(np.abs(d))),
                "cosine_centered": float(cos[local_j, i]),
                "generated_std": float(generated[j].std()),
                "train_std": float(real[i].std()),
            })
    return pd.DataFrame(rows).sort_values("mse").reset_index(drop=True)

nn_df = nearest_training_table(real, generated)
if len(nn_df):
    display(nn_df.head(20))
    out = OUTPUT_DIR / "n64_memorize_nearest_training_table.csv"
    nn_df.to_csv(out, index=False)
    print("wrote", out)
else:
    print("No generated samples loaded yet.")


In [ ]:
def plot_nearest_matches(real: np.ndarray, generated: np.ndarray, nn_df: pd.DataFrame, n: int = 8):
    if len(nn_df) == 0:
        print("no nearest-neighbor table")
        return
    n = min(n, len(nn_df))
    rows = nn_df.head(n)
    fig, axes = plt.subplots(n, 3, figsize=(8.5, 2.45 * n), squeeze=False)
    vals = np.concatenate([
        real[rows["nearest_train_index"].to_numpy(), 0].ravel(),
        generated[rows["generated_index"].to_numpy(), 0].ravel(),
    ])
    vmin, vmax = np.nanpercentile(vals, [1, 99])
    diff_abs = []
    for _, r in rows.iterrows():
        diff_abs.append(np.abs(generated[int(r.generated_index), 0] - real[int(r.nearest_train_index), 0]).ravel())
    dmax = float(np.nanpercentile(np.concatenate(diff_abs), 99)) if diff_abs else 1.0
    dmax = max(dmax, 1e-6)

    for axrow, (_, r) in zip(axes, rows.iterrows()):
        gj = int(r.generated_index)
        ti = int(r.nearest_train_index)
        g = generated[gj, 0]
        t = real[ti, 0]
        d = g - t
        axrow[0].imshow(g, origin="lower", cmap="viridis", vmin=vmin, vmax=vmax)
        axrow[0].set_title(f"gen {gj}")
        axrow[1].imshow(t, origin="lower", cmap="viridis", vmin=vmin, vmax=vmax)
        axrow[1].set_title(f"nearest train {ti}")
        axrow[2].imshow(d, origin="lower", cmap="coolwarm", vmin=-dmax, vmax=dmax)
        axrow[2].set_title(f"diff | mse={r.mse:.2e}, cos={r.cosine_centered:.4f}")
        for ax in axrow:
            ax.axis("off")
    fig.suptitle("closest generated-to-training matches")
    fig.tight_layout()
    out = OUTPUT_DIR / "n64_memorize_nearest_training_matches.png"
    fig.savefig(out, dpi=180, bbox_inches="tight")
    print("wrote", out)
    plt.show()

plot_nearest_matches(real, generated, nn_df, n=8)


## One-point and P(k) checks

These are not the primary memorization diagnostic, but they tell us whether the generated distribution has the right basic statistics.


In [ ]:
if len(generated):
    real_hist = field_histogram(real)
    gen_hist = field_histogram(generated)
    hist_l1 = float(np.mean(np.abs(np.asarray(real_hist["hist"]) - np.asarray(gen_hist["hist"]))))
    pk_summary = power_spectrum_summary(real, generated, nbins=30)
    summary = {
        "n_real": len(real),
        "n_generated": len(generated),
        "hist_l1": hist_l1,
        "real_std": real_hist["std"],
        "generated_std": gen_hist["std"],
        "std_ratio": gen_hist["std"] / max(real_hist["std"], 1e-30),
        **pk_summary,
    }
    display(pd.DataFrame([summary]))
else:
    print("No generated samples loaded yet.")


In [ ]:
if len(generated):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
    bins = np.asarray(real_hist["bin_edges"])
    centers = 0.5 * (bins[:-1] + bins[1:])
    axes[0].semilogy(centers, real_hist["hist"], color="black", lw=2.2, label="training")
    axes[0].semilogy(centers, gen_hist["hist"], color="tab:blue", lw=2.0, label="generated")
    axes[0].set_xlabel("normalized field value")
    axes[0].set_ylabel("density")
    axes[0].legend()
    axes[0].grid(alpha=0.2)

    pk_real, kbins = batch_power_spectra(real, nbins=30)
    pk_gen, _ = batch_power_spectra(generated, nbins=30)
    mean_real = np.nanmean(pk_real, axis=0)
    mean_gen = np.nanmean(pk_gen, axis=0)
    axes[1].loglog(kbins, mean_real, color="black", lw=2.2, label="training")
    axes[1].loglog(kbins, mean_gen, color="tab:blue", lw=2.0, marker="o", ms=3, label="generated")
    axes[1].set_xlabel("k bin")
    axes[1].set_ylabel("P(k)")
    axes[1].legend()
    axes[1].grid(alpha=0.2)
    fig.suptitle("N=64 no-aug memorization run: statistics")
    fig.tight_layout()
    out = OUTPUT_DIR / "n64_memorize_stats.png"
    fig.savefig(out, dpi=180, bbox_inches="tight")
    print("wrote", out)
    plt.show()
